
# Legal Prompt Engineering Playground

**Day 1 — AI Foundations · Practical 6 of 6 (main lab) · Companion to the "Prompt Engineering"
deck**

> **Running in Google Colab:** works fine on the default **CPU runtime** — every technique here
> calls a hosted API, no local model inference happens in this notebook. You just need an API
> key for whichever provider you choose in Section 0.

---

## Learning Objectives

By the end of this notebook, you will be able to apply every technique from the deck to a real
legal workflow:

0. **Anatomy of a Prompt** — explicitly identify Instruction, Context, Input, and Output Format
   inside a real legal prompt
1. **Zero-shot vs. few-shot** — classify a contract clause's type
2. **Chain-of-Thought (IRAC-style)** — reason through a short legal fact pattern step by step
3. **Step-back prompting** — retrieve legal principles before applying them to specifics
4. **System prompt & role assignment** — set a persistent "contract-review paralegal" persona
5. **Structured output — JSON mode and XML tags** — extract clause metadata reliably, in two
   different structured formats
6. **Prompt chaining** — a 3-stage pipeline mirroring a real legal memo workflow

## Setup Note

This notebook calls a hosted LLM API. **Three providers are supported out of the box** — pick
whichever your firm has access to by setting `PROVIDER` in Section 0:

| `PROVIDER` value | Service | Key needed |
|---|---|---|
| `"openai"` (default) | OpenAI | `OPENAI_API_KEY` |
| `"gemini"` | Google Gemini | `GEMINI_API_KEY` |
| `"groq"` | Groq | `GROQ_API_KEY` |

**Never hard-code an API key directly in a notebook cell.** In Google Colab, use the built-in
**Secrets** manager (the 🔑 key icon in the left sidebar) to store your key, then Section 0
reads it automatically — nothing gets written into the notebook file itself.

## Notebook Workflow

```mermaid
flowchart TD
    A["Contract clause /\nfact pattern"] --> Z["0. Anatomy of\na Prompt"]
    A --> B["1. Zero-shot vs\nFew-shot classification"]
    A --> C["2. Chain-of-Thought\n(IRAC reasoning)"]
    A --> D["3. Step-back\nprompting"]
    A --> E["4. System prompt +\nRole assignment"]
    A --> F1["5a. JSON structured\nextraction"]
    A --> F2["5b. XML-tagged\nstructured output"]
    A --> G["6. Prompt chaining\n(3-stage pipeline)"]

    G --> G1["Stage 1: Extract facts"]
    G1 --> G2["Stage 2: Draft memo section"]
    G2 --> G3["Stage 3: Validate against source"]



## Section 0 — API Client Setup (OpenAI / Gemini / Groq)

Pick a provider by setting `PROVIDER` below. The `ask()` helper function dispatches to whichever
client is configured — every technique later in this notebook calls `ask()` the same way
regardless of provider, so **the prompting techniques themselves are provider-agnostic.**


In [ ]:

# Install dependencies. Running in Google Colab: this cell installs everything needed for
# all three supported providers -- just run it (installing all three is harmless even if
# you only use one).
%pip install -q openai google-genai groq


In [ ]:

import os
import json

# --------------------------------------------------------------------------------
# Choose your provider here: "openai", "gemini", or "groq"
PROVIDER = "openai"
# --------------------------------------------------------------------------------

MODEL_BY_PROVIDER = {
    "openai": "gpt-4o-mini",
    "gemini": "gemini-3.6-flash",
    "groq": "llama-3.3-70b-versatile",
}
MODEL = MODEL_BY_PROVIDER[PROVIDER]
API_KEY_ENV_VAR = {
    "openai": "OPENAI_API_KEY",
    "gemini": "GEMINI_API_KEY",
    "groq": "GROQ_API_KEY",
}[PROVIDER]


def get_api_key(env_var_name):
    # In Google Colab, prefer the Secrets manager (the key icon in the left sidebar) --
    # this keeps the key out of the notebook file entirely. Falls back to a plain
    # environment variable for local/non-Colab use.
    try:
        from google.colab import userdata
        key = userdata.get(env_var_name)
        if key:
            return key
    except ImportError:
        pass  # not running in Colab

    return os.environ.get(env_var_name)


api_key = get_api_key(API_KEY_ENV_VAR)
if not api_key:
    raise ValueError(
        f"No API key found for {PROVIDER!r}. In Colab: add a secret named "
        f"{API_KEY_ENV_VAR!r} via the key icon in the left sidebar. "
        f"Locally: set the {API_KEY_ENV_VAR} environment variable."
    )

print(f"Provider: {PROVIDER}   Model: {MODEL}   API key loaded: {'yes' if api_key else 'no'}")


In [ ]:
# Build the provider-specific client, once, based on PROVIDER chosen above.

if PROVIDER == "openai":
    from openai import OpenAI
    _client = OpenAI(api_key=api_key)

elif PROVIDER == "gemini":
    from google import genai
    from google.genai import types as genai_types
    _client = genai.Client(api_key=api_key)

elif PROVIDER == "groq":
    from groq import Groq
    _client = Groq(api_key=api_key)


def ask(system_prompt, user_prompt, temperature=0.2, json_mode=False):
    # Unified chat call: same signature regardless of PROVIDER.
    #   system_prompt: persona/instructions that persist for this one call
    #   user_prompt:   the actual task/content for this call
    #   json_mode:     if True, ask the provider to guarantee valid JSON output
    if PROVIDER in ("openai", "groq"):
        kwargs = dict(
            model=MODEL,
            temperature=temperature,
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_prompt},
            ],
        )
        if json_mode:
            kwargs["response_format"] = {"type": "json_object"}

        response = _client.chat.completions.create(**kwargs)
        return response.choices[0].message.content

    elif PROVIDER == "gemini":
        config_kwargs = {"temperature": temperature, "system_instruction": system_prompt}
        if json_mode:
            config_kwargs["response_mime_type"] = "application/json"

        response = _client.models.generate_content(
            model=MODEL,
            contents=user_prompt,
            config=genai_types.GenerateContentConfig(**config_kwargs),
        )
        return response.text


print("ask() is ready. Try it:")
print(ask("You are a terse assistant.", "Say 'ready' and nothing else."))



---

## Technique 0 — Anatomy of a Prompt

Before running any technique, let's make the deck's **Anatomy of a Prompt** slide concrete:
every well-formed prompt has an **Instruction**, **Context**, **Input**, and **Output Format**.
Below we build the exact same underlying request TWO ways — once as one vague sentence, once
with all four parts explicitly labeled — and compare the results directly.


In [ ]:

review_target = (
    "The Vendor's total liability under this Agreement, whether in contract, tort, or "
    "otherwise, shall not exceed the amount paid by the Customer in the twelve (12) months "
    "preceding the claim."
)

# --- Vague: instruction only, no context/input/output-format separation ---
vague_prompt = f"Review this: {review_target}"

print("VAGUE PROMPT RESULT:\n")
print(ask("You are a legal assistant.", vague_prompt))


In [ ]:

# --- Same request, now explicitly structured into the 4 anatomy components ---
structured_prompt = (
    # CONTEXT: background the model needs
    "Context: You are reviewing this clause on behalf of the Customer (not the Vendor) "
    "in a commercial SaaS agreement.\n\n"
    # INSTRUCTION: the actual task, specific and unambiguous
    "Instruction: Identify whether this liability cap clause is favorable or unfavorable "
    "to the Customer, and explain why in plain business terms.\n\n"
    # INPUT: the actual content to act on
    f"Input clause: \"{review_target}\"\n\n"
    # OUTPUT FORMAT: explicit structure for the response
    "Output format: Respond in exactly two lines:\n"
    "Line 1: 'FAVORABLE' or 'UNFAVORABLE' to the Customer\n"
    "Line 2: one-sentence plain-English reason"
)

print("STRUCTURED PROMPT RESULT (Instruction + Context + Input + Output Format):\n")
print(ask("You are a legal assistant.", structured_prompt))



**Discussion:** the vague version leaves the model guessing at scope, audience, and desired
format -- responses will vary run to run. The structured version pins down all four anatomy
components explicitly, and should give a consistent, directly-usable two-line answer every
time. Every technique from here on builds on this same anatomy -- they just add specific
*strategies* (examples, reasoning steps, roles, schemas, chaining) on top of it.



---

## Technique 1 — Zero-Shot vs. Few-Shot Clause Classification

**Task:** given a contract clause, classify it into one of a fixed set of clause types
(mirroring the LEDGAR/CUAD-style taxonomy from the Model Landscape notebook).

We'll run the SAME clause through a **zero-shot** prompt (just an instruction, no examples) and
a **few-shot** prompt (a handful of labeled examples first), and compare reliability of the
output format.


In [ ]:

target_clause = (
    "The Receiving Party shall not disclose any Confidential Information of the Disclosing "
    "Party to any third party without the prior written consent of the Disclosing Party, "
    "except as required by applicable law."
)

# --- Zero-shot ---
zero_shot_prompt = (
    "Classify the following contract clause into exactly one category:\n"
    "Indemnification, Termination, Confidentiality, Governing Law, Force Majeure, or Arbitration.\n\n"
    f'Clause: "{target_clause}"\n\n'
    "Respond with only the category name."
)

print("ZERO-SHOT RESULT:")
print(ask("You are a contract clause classifier.", zero_shot_prompt))


In [ ]:

# --- Few-shot ---
few_shot_prompt = (
    "Classify a contract clause into exactly one category. Here are examples:\n\n"
    'Clause: "Either party may terminate this Agreement upon 30 days written notice."\n'
    "Category: Termination\n\n"
    'Clause: "The Contractor shall indemnify the Client against any third-party claims arising from its work."\n'
    "Category: Indemnification\n\n"
    'Clause: "This Agreement shall be governed by the laws of the State of New York."\n'
    "Category: Governing Law\n\n"
    "Now classify this clause:\n"
    f'Clause: "{target_clause}"\n'
    "Category:"
)

print("FEW-SHOT RESULT:")
print(ask("You are a contract clause classifier.", few_shot_prompt))



**Discussion:** run both prompts a few times (raise `temperature` slightly to see variance).
Few-shot prompting typically locks the model into the exact label format and category set you
provided, reducing the odds of an off-taxonomy answer — valuable when the output feeds directly
into a downstream classification pipeline.



---

## Technique 2 — Chain-of-Thought (IRAC-Style Legal Reasoning)

Legal reasoning already follows a canonical structure: **Issue, Rule, Application, Conclusion**
(IRAC). This is essentially Chain-of-Thought prompting, taught to law students long before LLMs
existed — which makes it a natural, high-signal way to demonstrate CoT to this audience.

We compare a direct-answer prompt against an explicit IRAC-structured prompt on the same fact
pattern.


In [ ]:

fact_pattern = (
    "A pedestrian was crossing a marked crosswalk when a delivery driver, who was "
    "looking at a phone, struck the pedestrian with their vehicle. The driver was traveling "
    "at the posted speed limit. The pedestrian suffered a broken leg."
)

# --- Direct answer, no structured reasoning ---
direct_prompt = (
    "Based on the following facts, was the driver negligent?\n\n"
    f"Facts: {fact_pattern}\n\n"
    "Give a one-sentence answer."
)

print("DIRECT ANSWER (no CoT):")
print(ask("You are a legal analyst.", direct_prompt))


In [ ]:

# --- IRAC / Chain-of-Thought ---
irac_prompt = (
    "Analyze the following facts using the IRAC method (Issue, Rule, Application, "
    "Conclusion). Work through each section explicitly before giving your conclusion.\n\n"
    f"Facts: {fact_pattern}\n\n"
    "Structure your response as:\n"
    "ISSUE: ...\n"
    "RULE: ...\n"
    "APPLICATION: ...\n"
    "CONCLUSION: ..."
)

print("IRAC / CHAIN-OF-THOUGHT ANSWER:")
print(ask(
    "You are a legal analyst providing preliminary issue-spotting analysis, not legal advice.",
    irac_prompt,
))



**Discussion:** the IRAC-structured response should show its reasoning about duty of care,
breach (distracted driving vs. speed compliance), causation, and damages explicitly — the same
benefit Chain-of-Thought provides for math word problems, just mapped onto the reasoning
structure lawyers already use every day.

> **Important:** outputs from this notebook are for training/demonstration purposes only and
> do not constitute legal advice — always emphasize this distinction when using LLMs for legal
> reasoning tasks in practice.



---

## Technique 3 — Step-Back Prompting

Step-back prompting asks the model to first answer a **higher-level, abstract question**, then
apply that answer to the specific case — rather than jumping straight to the specific analysis.

Using the same fact pattern, we first step back to ask about the general legal framework, then
apply it.


In [ ]:

# --- Step 1: the abstract, higher-level question ---
step_back_question = (
    "What are the essential elements a plaintiff must prove to establish a claim of negligence?"
)

print("STEP-BACK QUESTION RESULT (general principle):\n")
general_principle = ask("You are a legal analyst.", step_back_question)
print(general_principle)


In [ ]:

# --- Step 2: apply the retrieved general principle to the specific facts ---
apply_prompt = (
    "Using the following framework for negligence claims:\n\n"
    f"{general_principle}\n\n"
    "Apply this framework to the specific facts below and reach a conclusion:\n\n"
    f"Facts: {fact_pattern}"
)

print("APPLIED TO SPECIFIC FACTS (after step-back):\n")
print(ask(
    "You are a legal analyst providing preliminary issue-spotting analysis, not legal advice.",
    apply_prompt,
))



**Discussion:** compare this two-step result against the direct-answer result from Technique 2.
Step-back prompting tends to produce a more thorough, framework-grounded analysis because the
model explicitly retrieves the relevant legal elements *before* trying to match facts to them —
reducing the chance it skips an element (e.g. forgetting to address causation).



---

## Technique 4 — System Prompt Design & Role Assignment

Compare how the SAME user question gets answered under two different system-prompt personas:
a generic assistant vs. an explicitly-defined contract-review paralegal role with format rules.


In [ ]:

review_snippet = (
    "The Contractor's liability under this Agreement shall not exceed, in the "
    "aggregate, the total fees paid by the Client in the preceding six (6) months, except in "
    "cases of gross negligence or willful misconduct."
)

# --- Generic assistant persona ---
generic_system = "You are a helpful assistant."
generic_prompt = f"Review this clause and note any concerns: {review_snippet}"

print("GENERIC ASSISTANT PERSONA:\n")
print(ask(generic_system, generic_prompt))


In [ ]:

# --- Defined role + explicit constraints ---
paralegal_system = (
    "You are a senior contract-review paralegal at a corporate law firm. "
    "Review clauses for risk to the Client. Be direct and specific. Always structure your "
    "response as a numbered list of flagged issues, each with a brief risk rationale. Do not "
    "add pleasantries or disclaimers beyond a single closing note that this is not legal advice."
)

paralegal_prompt = f"Review this liability cap clause from the Client's perspective: {review_snippet}"

print("CONTRACT-REVIEW PARALEGAL PERSONA:\n")
print(ask(paralegal_system, paralegal_prompt))



**Discussion:** the role-assigned version should be noticeably more consistent in structure,
more specific in vocabulary (liability caps, carve-outs, gross negligence exceptions), and more
directly usable as a first-pass review note than the generic version — this is role assignment
combined with explicit output-structure rules, exactly as the deck recommends ("role alone can
drift; role + rules holds steady").



---

## Technique 5a — Structured Output: JSON Mode — Clause Metadata Extraction

A very common real workflow: pull structured metadata out of a clause (parties, obligation
type, key dates/numbers) so it can be stored in a database or spreadsheet, mirroring what a
CUAD-style annotation effort captures by hand. We call `ask(..., json_mode=True)`, which our
Section 0 helper translates into each provider's native "guaranteed valid JSON" mechanism
(OpenAI/Groq's `response_format={"type": "json_object"}`, Gemini's
`response_mime_type="application/json"`) — so this works unchanged no matter which `PROVIDER`
you picked.


In [ ]:

extraction_clause = (
    "The Contractor shall indemnify and hold harmless the Client from any "
    "third-party claims arising out of the Contractor's gross negligence, provided that the "
    "Client notifies the Contractor in writing within thirty (30) days of becoming aware of "
    "such claim."
)

extraction_system = "You are a precise legal-data extraction assistant. Respond only with valid JSON."

extraction_prompt = (
    "Extract the following fields from this clause as a JSON object:\n"
    '- "clause_type": the category of clause (e.g. Indemnification, Termination, etc.)\n'
    '- "obligated_party": which party has the obligation\n'
    '- "beneficiary_party": which party benefits from the obligation\n'
    '- "conditions": a list of any conditions or exceptions mentioned\n'
    '- "deadline_days": any numeric deadline mentioned, in days (or null if none)\n\n'
    f'Clause: "{extraction_clause}"\n\n'
    "Respond only with a JSON object matching this schema, no other text."
)

result = ask(
    extraction_system,
    extraction_prompt,
    json_mode=True,
)

print("RAW MODEL OUTPUT:")
print(result)

parsed = json.loads(result)
print("\nPARSED PYTHON DICT:")
print(json.dumps(parsed, indent=2))



**Discussion:** because the output is guaranteed-valid JSON, `json.loads()` succeeds reliably —
this is the difference between a demo and a production-usable extraction pipeline. Try changing
`extraction_clause` to a termination or confidentiality clause and re-run; the schema stays
fixed while the extracted values change.



---

## Technique 5b — Structured Output: XML Tags — Separating Reasoning From the Answer

JSON is ideal for fixed-schema data extraction. **XML tags** shine for a different case: when
you need to clearly delimit a *reasoning/scratch-work section* from a *final answer* inside one
longer response — so downstream code can pull out just the answer without parsing the whole
analysis. This combines nicely with Chain-of-Thought (Technique 2): let the model reason freely
inside `<reasoning>`, then commit to a clean answer inside `<answer>`.


In [ ]:

xml_fact_pattern = (
    "A commercial tenant's lease contains a clause stating rent 'shall increase annually "
    "by the greater of 3% or the CPI increase.' In a year where CPI increased by 6%, the "
    "landlord invoiced the tenant for a 6% increase. The tenant argues the clause caps the "
    "increase at 3%."
)

xml_prompt = (
    "Analyze whether the landlord's 6% rent increase is consistent with the lease clause "
    "below. Show your reasoning, then give a clear final answer.\n\n"
    f"Clause and facts: {xml_fact_pattern}\n\n"
    "Respond using EXACTLY this structure:\n"
    "<reasoning>your step-by-step analysis of what 'the greater of 3% or CPI' means here</reasoning>\n"
    "<answer>one or two sentence final conclusion</answer>"
)

xml_result = ask("You are a legal analyst providing preliminary issue-spotting analysis, not legal advice.", xml_prompt)
print("RAW MODEL OUTPUT (with XML tags):\n")
print(xml_result)


In [ ]:

import re

# Extract just the <answer> section programmatically -- this is the payoff of tagging:
# downstream code can grab exactly what it needs without parsing the full reasoning text.
answer_match = re.search(r"<answer>(.*?)</answer>", xml_result, re.DOTALL)
reasoning_match = re.search(r"<reasoning>(.*?)</reasoning>", xml_result, re.DOTALL)

print("EXTRACTED <answer> ONLY:\n")
print(answer_match.group(1).strip() if answer_match else "(no <answer> tag found)")

print("\nEXTRACTED <reasoning> ONLY (first 200 chars):\n")
if reasoning_match:
    print(reasoning_match.group(1).strip()[:200] + "...")



**Discussion:** notice how cleanly `re.search` pulls out just the `<answer>` block. In a real
pipeline, you'd store the reasoning for audit/review purposes but show the client (or route
downstream) only the `<answer>` — JSON mode couldn't easily represent "one field is free-form
long-form prose reasoning, another is a short conclusion" as cleanly as this tag-based approach
does.



---

## Technique 6 — Prompt Chaining — A 3-Stage Legal Memo Pipeline

The final and most involved exercise: break a realistic legal-memo drafting task into three
separate, chained prompts, mirroring an actual legal research workflow:

1. **Stage 1 (Extract)** — pull the key facts out of a case summary
2. **Stage 2 (Draft)** — draft a legal memo section based on those extracted facts
3. **Stage 3 (Validate)** — check the drafted memo section against the original source for
   unsupported claims

Each stage's output becomes the next stage's input — exactly the "Extract → Transform →
Validate" pattern from the deck.


In [ ]:

case_summary = (
    "In Smith v. Jones Logistics (2023), the plaintiff, a warehouse employee, "
    "slipped on an unmarked wet floor and suffered a wrist injury. The defendant company had a "
    "written policy requiring 'wet floor' signage within five minutes of any spill, but store "
    "video footage showed the spill had been present for over twenty minutes without a sign "
    "being placed. The plaintiff missed six weeks of work and incurred $18,000 in medical "
    "expenses."
)

# --- Stage 1: Extract key facts ---
stage1_prompt = (
    "Extract the key facts from this case summary as a concise bulleted list. "
    "Focus on: the parties, the incident, any relevant company policy, and damages.\n\n"
    f"Case summary: {case_summary}"
)

stage1_output = ask("You are a legal research assistant extracting key facts.", stage1_prompt)
print("STAGE 1 OUTPUT (extracted facts):\n")
print(stage1_output)


In [ ]:

# --- Stage 2: Draft a memo section using ONLY Stage 1's output as input ---
stage2_prompt = (
    'Using ONLY the facts below, draft a "Statement of Facts" section for an '
    "internal legal memo. Write in formal legal-memo prose, 3-5 sentences.\n\n"
    f"Extracted facts:\n{stage1_output}"
)

stage2_output = ask("You are a legal associate drafting an internal memo.", stage2_prompt)
print("STAGE 2 OUTPUT (drafted memo section):\n")
print(stage2_output)


In [ ]:

# --- Stage 3: Validate the draft against the ORIGINAL source, not Stage 1's summary ---
stage3_prompt = (
    "Compare the drafted memo section below against the original case summary. "
    "Identify any claims in the drafted section that are NOT supported by the original source, "
    "or any important facts from the original source that were omitted.\n\n"
    f"Original case summary:\n{case_summary}\n\n"
    f"Drafted memo section:\n{stage2_output}\n\n"
    "Respond with:\n"
    'UNSUPPORTED CLAIMS: (list any, or "None found")\n'
    'OMITTED FACTS: (list any, or "None found")'
)

stage3_output = ask("You are a meticulous legal fact-checker.", stage3_prompt)
print("STAGE 3 OUTPUT (validation against source):\n")
print(stage3_output)



**Discussion:** notice that Stage 3 is deliberately given the **original** case summary, not
Stage 1's extracted-facts summary — this is intentional. Validating against the true original
source (rather than against an intermediate, already-lossy summary) is what actually catches
drift/hallucination introduced anywhere earlier in the chain. This 3-stage pattern —
extract → draft → validate-against-source — generalizes to many legal AI workflows: contract
summarization, due-diligence review, deposition summary drafting, and more.



## Key Takeaways — Full Day 1 Recap

You've now hands-on exercised every technique from the Prompt Engineering deck, and every
concept from the four decks before it, entirely through a legal-domain lens:

0. **Anatomy of a Prompt** — Instruction, Context, Input, and Output Format, made explicit,
   turn a vague one-line ask into a consistently-answerable request
1. **Zero-shot vs. few-shot** — few-shot examples lock in output format reliability
2. **Chain-of-Thought (IRAC)** — structured step-by-step reasoning matches how lawyers already
   think, and measurably improves analysis completeness
3. **Step-back prompting** — retrieving the general framework before applying it to specifics
   reduces missed-element risk
4. **Role assignment + explicit rules** — the most reliable way to get consistently-formatted,
   domain-appropriate output
5. **JSON mode and XML tags** — two different structured-output strategies: JSON for
   fixed-schema data extraction, XML tags for separating reasoning from a final answer within
   one longer response
6. **Prompt chaining** — breaking complex legal-drafting workflows into inspectable,
   independently-validatable stages, with validation against the true original source

This notebook's techniques also work identically across **any of the three supported
providers** (OpenAI, Gemini, Groq) — swap `PROVIDER` in Section 0 and re-run; nothing else in
the notebook needs to change.

**This completes Day 1 — AI Foundations.** Every deck (Transformers 101, Tokenization,
Finetuning & KV Cache, Modern Model Landscape, Prompt Engineering) now has a hands-on,
legal-domain companion notebook. Day 2 continues with Tools & LLM Finetuning.

---

### A Closing Reminder

Every output generated in these notebooks — clause classifications, IRAC analyses, extracted
metadata, drafted memo sections — is a **demonstration of technique**, not a substitute for
attorney judgment. Production legal-AI workflows require human review, citation verification,
and appropriate confidentiality safeguards before any output reaches a client-facing document.
